# Revenue Forecasting

## Objective
Forecast next-month enterprise revenue using monthly finance signals, lagged behavior, rolling statistics, and model comparison.

The production implementation lives in `backend/app/ml/revenue_forecasting.py`.

## Business Context

Revenue forecasting answers a core executive question: what will happen next? The platform uses monthly finance data to estimate future revenue and quantify model performance with time-series validation.

## Architecture and Implementation Plan

1. Load the validated finance table from the processed dataset layer.
2. Convert the monthly series into a supervised learning frame with lag and rolling features.
3. Compare candidate regression models using time-series cross-validation.
4. Tune XGBoost with Optuna and register the best model.
5. Persist the comparison report, model registry entry, forecast output, and forecast plot.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name == 'notebooks' else current_dir
backend_root = project_root / 'backend'
sys.path.insert(0, str(backend_root))

from app.ml.revenue_forecasting import RevenueForecastingPaths, RevenueForecastingService

processed_dir = project_root / 'processed'
trained_models_dir = project_root / 'trained_models'
reports_dir = project_root / 'reports'
exports_dir = project_root / 'exports'
service = RevenueForecastingService(RevenueForecastingPaths(processed_dir=processed_dir, trained_models_dir=trained_models_dir, reports_dir=reports_dir, exports_dir=exports_dir))
service

## Train and Compare Models

The next cell trains the candidate models, tunes XGBoost, and registers the best model.

In [ ]:
comparison_frame, artifact, forecast_frame = service.train_and_register(n_trials=4)
comparison_frame

## Forecast Output

Review the forecast generated from the latest available finance feature row.

In [ ]:
forecast_frame

## Evaluation Visuals

Display the comparison metrics so model selection is transparent and reviewable.

In [ ]:
ax = comparison_frame.plot(kind='bar', x='model_name', y=['mae', 'rmse', 'r2'], figsize=(12, 4), title='Revenue Forecasting Model Comparison')
ax.set_xlabel('Model')
ax.set_ylabel('Metric Value')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

## Saved Artifacts

The notebook persists a model registry entry, forecast report, and plot to backend-consumable locations.

In [ ]:
artifact_summary = pd.DataFrame([
    {
        'model_name': artifact.model_name,
        'path': artifact.path,
        'mae': artifact.mae,
        'rmse': artifact.rmse,
        'r2': artifact.r2,
        'description': artifact.description,
    }
])
artifact_summary

## Conclusions

The platform can now produce a reproducible monthly revenue forecast, compare candidate models, persist the best performer, and emit forecast artifacts ready for API and dashboard integration.